In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

from utils import tfidf_search, bm25_search

In [2]:
documents = [
    "The cat sat on the mat and stared at the moon.",
    "Dogs are loyal companions that love to play fetch.",
    "Python is a popular programming language for data science.",
    "The stock market saw significant gains today.",
    "Cats and dogs are the most common household pets.",
    "Machine learning models require large amounts of training data.",
    "The weather today is sunny with a chance of rain later.",
    "Data science combines statistics, programming, and domain knowledge.",
]

query = "What programming language is used for data science?"

## Keyword Search: TF-IDF and BM25

Keyword search ranks documents by *lexical* overlap with the query's terms — blunter about meaning than embedding-based semantic search, but reliable for exact names/jargon, and often paired with vector search in hybrid retrieval. The toy `documents` corpus above is just enough to prove out the two most common scoring functions.

### TF-IDF (Term Frequency – Inverse Document Frequency)

For a term $t$, document $d$, and corpus $D$:

$$
\text{tfidf}(t, d, D) = \text{tf}(t, d) \times \text{idf}(t, D)
$$

$$
\text{tf}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}
\qquad\qquad
\text{idf}(t, D) = \log\left(\frac{N}{1 + n_t}\right)
$$

where $f_{t,d}$ is how many times $t$ appears in $d$, $N = |D|$ is the number of documents in the corpus, and $n_t$ is the number of documents containing $t$. A document's score for a multi-term query is the cosine similarity between the query's TF-IDF vector and the document's TF-IDF vector.

### BM25 (Best Matching 25)

BM25 refines TF-IDF with two extra ideas: term-frequency **saturation** (the 5th occurrence of a word matters much less than the 1st) and **document-length normalization** (a long document naturally contains more term repeats, so it shouldn't win purely on length). For query terms $q_1, \dots, q_n$ and document $d$:

$$
\text{BM25}(d, q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, d) \cdot (k_1 + 1)}{f(q_i, d) + k_1 \cdot \left(1 - b + b \cdot \dfrac{|d|}{\text{avgdl}}\right)}
$$

$$
\text{IDF}(q_i) = \log\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)
$$

- $f(q_i, d)$: count of term $q_i$ in document $d$
- $|d|$: length of $d$ in tokens; $\text{avgdl}$: average document length across the corpus
- $n(q_i)$: number of documents containing $q_i$
- $k_1$ (typically 1.2–2.0) controls how quickly extra occurrences saturate; $b$ (typically 0.75) controls how strongly length is normalized

In [3]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

for text, score in tfidf_search(query, tfidf_vectorizer, tfidf_matrix, documents):
    print(f"{score:.4f} | {text}")

0.8357 | Python is a popular programming language for data science.
0.3527 | Data science combines statistics, programming, and domain knowledge.
0.1119 | The weather today is sunny with a chance of rain later.


In [4]:
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

for text, score in bm25_search(query, bm25, documents):
    print(f"{score:.4f} | {text}")

6.7811 | Python is a popular programming language for data science.
1.0116 | Data science combines statistics, programming, and domain knowledge.
0.8746 | The weather today is sunny with a chance of rain later.
